# Week 1: Gaia Stellar Data Exploration

**Goal:** Query real star data from the Gaia DR3 catalog, clean it, engineer features, and plot the Hertzsprung-Russell (HR) diagram.

**What is Gaia?**  
Gaia is a European Space Agency satellite that has measured the positions, distances, and brightness of ~1.8 billion stars with unprecedented precision. The data is publicly available and queried using ADQL — a dialect of SQL for astronomical archives.

**What is the HR Diagram?**  
A scatter plot of a star's color (temperature proxy) vs. its absolute magnitude (true brightness). Stars don't scatter randomly — they cluster into sequences that reveal their evolutionary stage:  
- **Main sequence** — stars fusing hydrogen (like our Sun)  
- **Red giant branch** — stars that have exhausted hydrogen in their core  
- **White dwarfs** — stellar remnants, very hot but dim  

---

**Notebook flow:**  
`Setup → Query Gaia → Explore Raw Data → Clean → Feature Engineering → Visualize`

## Setup

Add the `src/` directory to the Python path so we can import our reusable modules.  
This is needed because notebooks run from the `notebooks/` directory, not the project root.

In [ ]:
import sys
import os

# Point Python at the project root so 'from src import ...' works
sys.path.insert(0, os.path.abspath(".."))

from src.data_fetch import fetch_gaia_sample
from src.data_clean import clean_sample, add_features
from src import visualize

## Step 1: Query the Gaia Archive

We fetch 10,000 stars from the `gaiadr3.gaia_source` table.  

**Why only 10,000?**  
The full DR3 table has ~1.8 billion rows. Limiting to 10k keeps the query fast and the notebook responsive while still showing all the major stellar populations on the HR diagram.

**Why `WHERE parallax IS NOT NULL`?**  
Parallax is essential for computing distance (and from that, absolute magnitude). Stars without a measured parallax can't be placed on the HR diagram.

**Key columns:**
| Column | Meaning |
|--------|---------|
| `parallax` | Angular shift in milliarcseconds — used to derive distance |
| `parallax_error` | Measurement uncertainty on parallax |
| `phot_g_mean_mag` | Apparent brightness in the G (green) band |
| `bp_rp` | Color index: blue minus red. Low = hot blue star, high = cool red star |

In [ ]:
df = fetch_gaia_sample(top_n=10000)
print(f"Rows returned: {len(df):,}")
df.head()

## Step 2: Explore the Raw Data

Before cleaning, look at the shape of the parallax distribution.  

**What to expect:**  
Most stars are distant (small parallax). There's a long tail of nearby stars with large parallax values. Negative values are measurement artifacts — we'll remove those in the cleaning step.

In [ ]:
# Quick summary stats before any filtering
print(df[["parallax", "parallax_error", "phot_g_mean_mag", "bp_rp"]].describe())

In [ ]:
visualize.plot_parallax_hist(df)

## Step 3: Clean the Data

We apply three quality cuts — see `src/data_clean.py` for the full explanation of each.

1. **`parallax > 0`** — removes unphysical negative values from measurement noise  
2. **`parallax / parallax_error > 5`** — keeps only stars where distance is reliable (SNR ≥ 5)  
3. **`bp_rp` not null** — removes stars with no color measurement (can't go on HR diagram)

In [ ]:
clean = clean_sample(df)

## Step 4: Feature Engineering

We derive two columns that aren't in the raw Gaia data but are essential for the HR diagram:

**Distance in parsecs:**  
`distance_pc = 1000 / parallax_mas`  
A parsec is 3.26 light-years. Parallax in milliarcseconds inverts directly to distance in parsecs.

**Absolute magnitude:**  
`M = m - 5 * log10(d / 10)`  
This is the **distance modulus** formula. It corrects apparent brightness for distance, giving the intrinsic luminosity of the star. The reference point (d=10 pc) is the astronomy standard.

In [ ]:
clean = add_features(clean)
print(clean[["bp_rp", "absolute_mag", "distance_pc"]].describe())

## Step 5: Visualize

### 5a. HR Diagram (density map)

The hex-bin plot uses log-color to show structure across wide count ranges.  
The **y-axis is inverted** — astronomy convention: bright stars (low magnitude number) appear at the top.

**What to look for:**
- A dense diagonal band from top-left to bottom-right → **main sequence**
- A clump at upper-right (bright + red) → **red giants**
- A sparse cluster at lower-left (dim + blue) → **white dwarfs**

In [ ]:
visualize.plot_hr_density(clean)

### 5b. Absolute Magnitude Distribution

Shows the spread of intrinsic brightnesses in our sample.  
The peak is dominated by main-sequence stars around M ≈ 4–6 (similar to the Sun at M ≈ 4.8).

In [ ]:
visualize.plot_abs_mag_hist(clean)

### 5c. Distance vs. Brightness — Malmquist Bias

**Malmquist bias** is a selection effect: at large distances, only intrinsically bright stars are above the detection limit. Faint stars far away are invisible to Gaia (or excluded by our SNR cut).

In this plot you'll see a clear upper envelope — as distance increases, the minimum detectable brightness decreases. The lower-right region (faint + distant) is empty, not because those stars don't exist, but because we can't see them from here.

In [ ]:
visualize.plot_distance_vs_mag(clean)

---

## Summary

| Step | What we did | Why |
|------|-------------|-----|
| Query | Fetched 10k stars from Gaia DR3 via ADQL | Real observational data |
| Clean | Removed negative parallax, low-SNR stars, missing colors | Unreliable data corrupts the HR diagram |
| Features | Computed distance and absolute magnitude | Needed for the HR diagram axes |
| Visualize | HR density map, magnitude histogram, Malmquist plot | Understand stellar populations and sample biases |

**Next:** `week2_kmeans_clustering.ipynb` — use K-Means to label stellar populations without any predefined categories.